## Tratamento dos dados

Nesse notebook os dados são tratados de forma a tornar possível a análise

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from utils.load_csv import load_csv
from utils.load_shp import load_shp
from utils.save_csv import save_csv
from utils.save_shp import save_shp

In [2]:
df_resultados_2026 = load_csv('resultados_2026.csv')

df_eleitorado_sp_2026 = load_csv('eleitorado_sp_2026.csv')

df_observasampa = load_csv('observasampa.csv')

gdf_distritos = load_shp('distritos.shp')

In [3]:
# Resultado da eleição presidencial brasileira por seção eleitoral do país
# Seleciona as variáveis de interesse, filtra apenas o segundo turno na cidade de São Paulo

variaveis = ['NR_TURNO',
             'SG_UF',
             'CD_MUNICIPIO',
             'NM_MUNICIPIO',
             'NR_ZONA',
             'NR_SECAO',
             'CD_CARGO',
             'DS_CARGO',
             'NR_VOTAVEL',
             'NM_VOTAVEL',
             'QT_VOTOS',
             'NR_LOCAL_VOTACAO',
             'NM_LOCAL_VOTACAO',
             'DS_LOCAL_VOTACAO_ENDERECO']

df_resultados_2026_filtrado = (df_resultados_2026.loc[
    (df_resultados_2026['NR_TURNO'] == 2) &
    (df_resultados_2026['NM_MUNICIPIO'] == 'SÃO PAULO'),
    variaveis].copy())

In [4]:
# Eleitorado por local de votação do estado de São Paulo
# Seleciona as variáveis de interesse, filtra o 2º turno na cidade de São Paulo, valida e completa as coordenadas geográficas dos locais de votação

COD_MUNICIPIO_SP = 71072        
COD_IBGE_SP = "3550308"         
TURNO = 2
CHAVE_LOCAL = ["NR_ZONA", "NR_LOCAL_VOTACAO"]

colunas = [
    "NR_TURNO", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO",
    "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO", "DS_TIPO_LOCAL",
    "DS_ENDERECO", "NM_BAIRRO", "NR_CEP", "NR_TELEFONE_LOCAL",
    "NR_LATITUDE", "NR_LONGITUDE", "DS_SITU_LOCAL_VOTACAO", "QT_ELEITOR_SECAO",
]

df = df_eleitorado_sp_2026[colunas].copy()

mask = (
    (pd.to_numeric(df["CD_MUNICIPIO"], errors="coerce") == COD_MUNICIPIO_SP)
    & (pd.to_numeric(df["NR_TURNO"], errors="coerce") == TURNO)
)
sp = df.loc[mask].copy()
assert len(sp) > 0, "Filtro de turno/município não retornou linhas"

for c in ["NR_TURNO", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "NR_LOCAL_VOTACAO"]:
    sp[c] = pd.to_numeric(sp[c], errors="coerce").astype("Int64")

assert sp[["NR_ZONA", "NR_SECAO", "NR_LOCAL_VOTACAO"]].notna().all().all(), \
    "Há zona/seção/local nulos"

dup = sp.duplicated(["NR_ZONA", "NR_SECAO"], keep=False)
assert not dup.any(), f"{dup.sum()} linhas duplicadas por (zona, seção)"

for col, novo in [("NR_LATITUDE", "lat"), ("NR_LONGITUDE", "lon")]:
    sp[novo] = pd.to_numeric(
        sp[col].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )

sem_coord = sp["lat"].isin([-1, 0]) | sp["lon"].isin([-1, 0])
sp.loc[sem_coord, ["lat", "lon"]] = np.nan

sp["NR_CEP"] = (
    pd.to_numeric(sp["NR_CEP"], errors="coerce").astype("Int64").astype(str).str.zfill(8)
    .where(sp["NR_CEP"].notna())
)
sp["QT_ELEITOR_SECAO"] = pd.to_numeric(sp["QT_ELEITOR_SECAO"], errors="coerce")
sp = sp.sort_values(["NR_ZONA", "NR_LOCAL_VOTACAO", "NR_SECAO"])

def primeiro_valido(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

locais_votacao = (
    sp.groupby(CHAVE_LOCAL)
    .agg(
        nome_local=("NM_LOCAL_VOTACAO", primeiro_valido),
        tipo_local=("DS_TIPO_LOCAL", primeiro_valido),
        endereco=("DS_ENDERECO", primeiro_valido),
        bairro=("NM_BAIRRO", primeiro_valido),
        cep=("NR_CEP", primeiro_valido),
        telefone=("NR_TELEFONE_LOCAL", primeiro_valido),
        lat=("lat", primeiro_valido),
        lon=("lon", primeiro_valido),
        ativo=("DS_SITU_LOCAL_VOTACAO", lambda s: (s == "ATIVO").any()),
        qt_secoes=("NR_SECAO", "nunique"),
        qt_eleitores=("QT_ELEITOR_SECAO", "sum"),
    )
    .reset_index()
    .rename(columns={"NR_ZONA": "zona", "NR_LOCAL_VOTACAO": "nr_local"})
)

for c in ["zona", "nr_local", "qt_secoes", "qt_eleitores"]:
    locais_votacao[c] = locais_votacao[c].astype(int)

locais_votacao["lat"] = pd.to_numeric(locais_votacao["lat"], errors="coerce").astype(float)
locais_votacao["lon"] = pd.to_numeric(locais_votacao["lon"], errors="coerce").astype(float)

locais_votacao["id_local"] = [
    f"{z:03d}-{n:04d}" for z, n in zip(locais_votacao["zona"], locais_votacao["nr_local"])
]
assert locais_votacao["id_local"].is_unique

locais_votacao = locais_votacao[[
    "id_local", "zona", "nr_local", "nome_local", "tipo_local", "endereco",
    "bairro", "cep", "telefone", "lat", "lon", "ativo", "qt_secoes", "qt_eleitores",
]].sort_values(["zona", "nr_local"]).reset_index(drop=True)

distritos_sp = gdf_distritos.loc[gdf_distritos["CD_MUN"] == COD_IBGE_SP, ["CD_DIST", "geometry"]]
assert distritos_sp.crs is not None, "A malha de distritos está sem CRS"

def ids_fora_de_sp(df_locais):
    """Retorna os id_local com coordenada que não cai em nenhum distrito de São Paulo."""
    com = df_locais.dropna(subset=["lat", "lon"])
    pts = gpd.GeoDataFrame(
        com[["id_local"]],
        geometry=gpd.points_from_xy(com["lon"], com["lat"]),
        crs="EPSG:4326",                       
    ).to_crs(distritos_sp.crs) # type: ignore                
    dentro = gpd.sjoin(pts, distritos_sp, how="inner", predicate="intersects")["id_local"]
    return set(com["id_local"]) - set(dentro)

fora = ids_fora_de_sp(locais_votacao)
if fora:
    print(f"{len(fora)} locais com coordenada do TSE fora de SP (serão tratados como ausentes):")
    print(locais_votacao.loc[locais_votacao["id_local"].isin(fora),
                             ["id_local", "nome_local", "bairro", "lat", "lon"]].to_string(index=False))
    locais_votacao.loc[locais_votacao["id_local"].isin(fora), ["lat", "lon"]] = np.nan

coords_manuais = {
    "003-1422": (-23.548233094486427, -46.615741162879324),  # Fundação Casa - Complexo Brás
    "003-1589": (-23.532442257354216, -46.59493409171579),   # Fundação Casa - Complexo Belém
    "248-1953": (-23.54522117359112,  -46.46239947637138),   # UI Itaquera
    "250-2267": (-23.57320104217256,  -46.70176366287856),   # CDP III Pinheiros
    "254-1597": (-23.527715829940284, -46.59310916288043),   # Fundação Casa - Nova Vida
    "254-1600": (-23.52747029260085,  -46.59249930520795),   # Fundação Casa - Ouro Preto
    "254-1910": (-23.504315928392757, -46.620843478225225),  # Penitenciária Feminina Sant'Ana
    "256-1830": (-23.463336561148697, -46.62083556288296),   # Presídio PM Romão Gomes
    "257-2224": (-23.588989161585978, -46.590668520549954),  # CDP Vila Independência
    "258-1899": (-23.60697179732972,  -46.691409076368636),  # EMEI Isabel Colombo
    "320-1619": (-23.630919411871282, -46.644143934039455),  # CEI Indir Jabaquara
    "325-2178": (-23.473277177450242, -46.73605762055472),   # Internato Pirituba
    "328-2410": (-23.638792749804978, -46.75015076869573),   # EMEI Carlos de Laet
    "403-1635": (-23.472571922413252, -46.7552062763743),    # EMEF Chácara Turística
}

ids_sem = set(locais_votacao.loc[locais_votacao[["lat", "lon"]].isna().any(axis=1), "id_local"])

faltando = ids_sem - set(coords_manuais)
assert not faltando, f"Locais sem coordenada e sem entrada manual: {sorted(faltando)}"

sobrando = set(coords_manuais) - ids_sem
if sobrando:
    print(f"Entradas manuais não usadas (local ausente no 2º turno ou com coordenada válida): {sorted(sobrando)}")

for id_local in ids_sem:
    lat, lon = coords_manuais[id_local]
    m = locais_votacao["id_local"] == id_local
    locais_votacao.loc[m, ["lat", "lon"]] = [lat, lon]

assert locais_votacao[["lat", "lon"]].notna().all().all()
fora_final = ids_fora_de_sp(locais_votacao)
assert not fora_final, f"Coordenadas fora de SP após ajuste manual: {sorted(fora_final)}"

print(f"{len(locais_votacao)} locais de votação | {locais_votacao['qt_eleitores'].sum():,} eleitores (2º turno)")

secoes_cad = (
    sp[["NR_ZONA", "NR_SECAO", "NR_LOCAL_VOTACAO", "QT_ELEITOR_SECAO"]]
    .rename(columns={"NR_ZONA": "zona", "NR_SECAO": "secao",
                     "NR_LOCAL_VOTACAO": "nr_local", "QT_ELEITOR_SECAO": "qt_eleitores"})
    .astype({"zona": int, "secao": int, "nr_local": int})
    .reset_index(drop=True)
)

secoes_cad["id_local"] = [f"{z:03d}-{n:04d}" for z, n in zip(secoes_cad["zona"], secoes_cad["nr_local"])]

assert not secoes_cad.duplicated(["zona", "secao"]).any()
assert set(secoes_cad["id_local"]) == set(locais_votacao["id_local"])

2057 locais de votação | 9,135,407 eleitores (2º turno)


In [5]:
# Indicadores socioeconômicos do município de São Paulo por distrito
# Seleciona as variáveis de interesse, filtra apenas os distritos e o período de 2024

variaveis = ["01.03.02 Quantidade de famílias beneficiárias do Programa Bolsa Família",
             "População total"]

df_observasampa_filtrado = df_observasampa[df_observasampa["Nome"].isin(variaveis)
                                           & df_observasampa["Região"].str.endswith("(Distrito)")
                                           & (df_observasampa["Período"] == 2024)].copy()

df_observasampa_filtrado["Resultado"] = pd.to_numeric(df_observasampa_filtrado["Resultado"])

df_observasampa_filtrado = (df_observasampa_filtrado
                            .pivot(index="Região", columns="Nome", values="Resultado")
                            .rename(columns={"01.03.02 Quantidade de famílias beneficiárias do Programa Bolsa Família": "familias_pbf",
                                             "População total": "pop_total"})
                            .rename_axis(columns=None)     
                            .reset_index()
                            .rename(columns={"Região": "distrito"}))

df_observasampa_filtrado["distrito"] = df_observasampa_filtrado["distrito"].str.replace(r"\s*\(Distrito\)$", "", regex=True)

In [6]:
# Malha territorial dos limites administrativos dos distritos do estado de São Paulo
# Seleciona as variáveis de interesse e filtra apenas os distritos da cidade de São Paulo

gdf_distritos_filtrado = gdf_distritos.loc[gdf_distritos["CD_MUN"] == "3550308",
                                           ["CD_DIST", "NM_DIST", "geometry"]].reset_index(drop=True)

In [7]:
save_csv(df_resultados_2026_filtrado, 'resultados_2026_filtrado.csv')

save_csv(locais_votacao, 'locais_votacao.csv')

save_csv(secoes_cad, "secoes_cadastro.csv")

save_csv(df_observasampa_filtrado, 'observasampa_filtrado.csv')

save_shp(gdf_distritos_filtrado, 'distritos_filtrado.shp')

Base salva em data\resultados_2026_filtrado.csv
Base salva em data\locais_votacao.csv
Base salva em data\secoes_cadastro.csv
Base salva em data\observasampa_filtrado.csv
Shapefile salvo em data\distritos_filtrado.shp
